# Deadline markets — step 1: contract and rules audit

We are comparing **the same question with different deadlines**, starting with six contracts:

- Strait of Hormuz traffic returns to normal: October, November and December 2026.
- Russia–Ukraine ceasefire agreement: October, November and December 2026.

This notebook identifies the contracts, preserves their full rules, maps **Yes/No token IDs**, and records which comparisons need care. It does **not** download price histories/trades or place orders.

**Run in VS Code:** select this repository's `.venv` Python kernel, then **Run All**. The first run makes only four public Gamma requests (with a one-second delay). Subsequent runs reuse the saved responses unless you set `REFRESH_METADATA = True`. All generated data stays in the Git-ignored `data/` directory.

Your existing `03_time_testing.ipynb` is left unchanged. This audit can supply its contract IDs, but does not certify a trading strategy.

API reference: [Polymarket: get event by slug](https://docs.polymarket.com/api-reference/events/get-event-by-slug).


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
from urllib.error import HTTPError
from uuid import uuid4
import hashlib
import html
import json
import re
import sys
import time

import pandas as pd
from IPython.display import HTML, display
from polytrader.data.client import DataError, PolymarketClient

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pyproject.toml").exists() and (p / "src/polytrader").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the polytrader repository.")

BASE = ROOT / "data/research/deadline_markets/step1"
CACHE = BASE / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.max_columns", 16)
print("Python:", sys.executable)
print("Repository:", ROOT)
print("Data directory:", BASE)


Python: c:\Users\kpast\OneDrive\Dokumenty\Polytrader\polytrader\.venv\Scripts\python.exe
Repository: c:\Users\kpast\OneDrive\Dokumenty\Polytrader\polytrader
Data directory: c:\Users\kpast\OneDrive\Dokumenty\Polytrader\polytrader\data\research\deadline_markets\step1


## 1. The shortlist

Keep this pilot at six contracts. September is already past and can be added later for a historical study. The US–Iran **continues through** family is excluded for now because its deadline direction is reversed.

An **event** can contain several **markets**. We select exact market slugs, never the first market or a fuzzy text match.


In [2]:
ALLOW_NETWORK = True        # Set False to require cached data only.
REFRESH_METADATA = False    # True explicitly fetches four fresh event records.
REQUEST_DELAY_SECONDS = 1.0
MAX_ATTEMPTS = 3

HORMUZ_EVENTS = {
    "2026-10-31": "strait-of-hormuz-traffic-returns-to-normal-by-october-31-20260810151043583",
    "2026-11-30": "strait-of-hormuz-traffic-returns-to-normal-by-november-30-20260810151158765",
    "2026-12-31": "strait-of-hormuz-traffic-returns-to-normal-by-december-31",
}
UKRAINE_EVENT = "russia-x-ukraine-ceasefire-agreement-by"
MONTH_ENDS = [("october", 31), ("november", 30), ("december", 31)]

SHORTLIST = [
    dict(family="hormuz_normalisation", deadline_date=date,
         event_slug=slug, market_slug=slug)
    for date, slug in HORMUZ_EVENTS.items()
] + [
    dict(family="russia_ukraine_agreement",
         deadline_date=f"2026-{month_number:02d}-{day:02d}",
         event_slug=UKRAINE_EVENT,
         market_slug=f"russia-x-ukraine-ceasefire-agreement-by-{month}-{day}-2026")
    for month_number, (month, day) in enumerate(MONTH_ENDS, start=10)
]
display(pd.DataFrame(SHORTLIST)[["family", "deadline_date", "market_slug"]])


,family,deadline_date,market_slug
0,hormuz_normalisation,2026-10-31,strait-of-hormuz-traffic-returns-to-normal-by-october-31-20260810151043583
1,hormuz_normalisation,2026-11-30,strait-of-hormuz-traffic-returns-to-normal-by-november-30-20260810151158765
2,hormuz_normalisation,2026-12-31,strait-of-hormuz-traffic-returns-to-normal-by-december-31
3,russia_ukraine_agreement,2026-10-31,russia-x-ukraine-ceasefire-agreement-by-october-31-2026
4,russia_ukraine_agreement,2026-11-30,russia-x-ukraine-ceasefire-agreement-by-november-30-2026
5,russia_ukraine_agreement,2026-12-31,russia-x-ukraine-ceasefire-agreement-by-december-31-2026


## 2. Fetch and preserve the source records

The cache contains the unmodified event response plus its URL and retrieval time. A cached record is a historical snapshot, **not current live status**. Refreshing preserves old snapshots in previously exported run folders.

Retries are bounded. A persistent HTTP error stops the notebook instead of silently skipping a contract.


In [3]:
client = PolymarketClient(timeout=20)

def atomic_json(path, payload):
    temp = path.with_name(path.name + "." + uuid4().hex + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)

def event_snapshot(slug):
    path = CACHE / f"{slug}.json"
    if path.exists() and not REFRESH_METADATA:
        snapshot = json.loads(path.read_text(encoding="utf-8"))
        if snapshot["event"]["slug"] != slug:
            raise ValueError(f"Wrong event in cache: {path}")
        print("Cached:", slug, "| fetched", snapshot["fetched_at_utc"])
        return snapshot
    if not ALLOW_NETWORK:
        raise RuntimeError(f"Fresh metadata needed for {slug}; enable ALLOW_NETWORK.")
    for attempt in range(1, MAX_ATTEMPTS + 1):
        time.sleep(REQUEST_DELAY_SECONDS)
        try:
            event = client.get_event(slug)
            if event.get("slug") != slug or not isinstance(event.get("markets"), list):
                raise ValueError(f"Unexpected event response for {slug}")
            snapshot = {
                "source_url": f"https://gamma-api.polymarket.com/events/slug/{slug}",
                "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
                "event": event,
            }
            atomic_json(path, snapshot)
            print("Fetched:", slug)
            return snapshot
        except DataError as exc:
            cause = exc.__cause__
            retryable = (
                not isinstance(cause, HTTPError)
                or cause.code in {408, 429, 500, 502, 503, 504}
            )
            if not retryable or attempt == MAX_ATTEMPTS:
                raise
            delay = min(2 ** attempt, 8)
            print(f"Request failed ({attempt}/{MAX_ATTEMPTS}); retry in {delay}s: {exc}")
            time.sleep(delay)

event_slugs = list(dict.fromkeys(spec["event_slug"] for spec in SHORTLIST))
snapshots = {slug: event_snapshot(slug) for slug in event_slugs}
print(f"Ready: {len(snapshots)} event records for {len(SHORTLIST)} exact contracts.")


Cached: strait-of-hormuz-traffic-returns-to-normal-by-october-31-20260810151043583 | fetched 2026-10-03T13:35:14.780345+00:00
Cached: strait-of-hormuz-traffic-returns-to-normal-by-november-30-20260810151158765 | fetched 2026-10-03T13:35:15.822882+00:00
Cached: strait-of-hormuz-traffic-returns-to-normal-by-december-31 | fetched 2026-10-03T13:35:16.856021+00:00
Cached: russia-x-ukraine-ceasefire-agreement-by | fetched 2026-10-03T13:35:17.899851+00:00
Ready: 4 event records for 6 exact contracts.


## 3. IDs, dates and full rules

Token IDs are large **strings**, not floating-point numbers. Match tokens to outcome labels rather than assuming Yes is always first.

We keep three different concepts separate:

- **Rule deadline date:** the date in the contract question/rules.
- **Rule cutoff:** an explicit time and timezone, if the rules provide one.
- **API end date:** Gamma's metadata, saved separately; it is not a substitute for reading the rules.

Hormuz specifies an observation date and publication/revision provisions, but no exact intraday observation cutoff. We leave its rule cutoff blank rather than inventing one. Ukraine explicitly uses 11:59 PM Eastern Time (including daylight saving).


In [4]:
def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def json_list(value, field):
    decoded = json.loads(value) if isinstance(value, str) else value
    if not isinstance(decoded, list):
        raise ValueError(f"{field} is not a JSON list")
    return decoded

def outcome_tokens(outcomes, token_ids):
    outcomes = json_list(outcomes, "outcomes")
    token_ids = json_list(token_ids, "clobTokenIds")
    labels = [str(outcome).strip().casefold() for outcome in outcomes]
    if len(labels) != 2 or set(labels) != {"yes", "no"} or len(token_ids) != 2:
        raise ValueError("Expected one Yes and one No outcome and two tokens.")
    if not all(isinstance(token, str) and token.isdigit() for token in token_ids):
        raise ValueError("Token IDs must be decimal strings, never floats.")
    if len(set(token_ids)) != 2:
        raise ValueError("Outcome token IDs must be distinct.")
    return dict(zip(labels, token_ids))

def rule_template(rules, deadline_label):
    # Only replace this contract's exact date. Do NOT remove thresholds,
    # durations, publication lags, source names, or arbitrary numbers.
    return re.sub(r"\s+", " ", rules.replace(deadline_label, "<DEADLINE>")).strip()

records = []
token_records = []
for spec in SHORTLIST:
    snapshot = snapshots[spec["event_slug"]]
    event = snapshot["event"]
    matches = [m for m in event["markets"] if m.get("slug") == spec["market_slug"]]
    if len(matches) != 1:
        raise ValueError(f"Expected one exact match: {spec['market_slug']}; found {len(matches)}")
    market = matches[0]
    rules = market.get("description", "").strip()
    if not rules:
        raise ValueError(f"Missing market-level rules: {spec['market_slug']}")
    token_map = outcome_tokens(market.get("outcomes"), market.get("clobTokenIds"))
    deadline = pd.Timestamp(spec["deadline_date"])
    deadline_label = f"{deadline.strftime('%B')} {deadline.day}, {deadline.year}"
    question = market.get("question", "")
    question_date = f"{deadline.strftime('%B')} {deadline.day}"
    if question_date not in question:
        raise ValueError(f"Question does not match the configured deadline: {question}")
    template = rule_template(rules, deadline_label)
    cutoff = None
    if spec["family"] == "russia_ukraine_agreement":
        if "11:59 PM ET" not in rules:
            raise ValueError("Ukraine cutoff wording changed; review before converting time.")
        cutoff = (
            pd.Timestamp(spec["deadline_date"] + " 23:59:00")
            .tz_localize("America/New_York").tz_convert("UTC").isoformat()
        )
    elif deadline_label not in rules:
        raise ValueError("Hormuz rules do not mention the configured full date.")
    row = {
        **spec,
        "question": question,
        "event_id": str(event["id"]),
        "market_id": str(market["id"]),
        "condition_id": str(market.get("conditionId", "")),
        "yes_token_id": token_map["yes"],
        "no_token_id": token_map["no"],
        "event_url": f"https://polymarket.com/event/{spec['event_slug']}",
        "market_url": f"https://polymarket.com/event/{spec['event_slug']}?marketSlug={spec['market_slug']}",
        "source_url": snapshot["source_url"],
        "fetched_at_utc": snapshot["fetched_at_utc"],
        "market_created_at_utc": market.get("createdAt"),
        "api_start_date_utc": market.get("startDate"),
        "api_end_date_utc": market.get("endDate"),
        "rule_cutoff_utc": cutoff,
        "active_at_snapshot": market.get("active"),
        "closed_at_snapshot": market.get("closed"),
        "accepting_orders_at_snapshot": market.get("acceptingOrders"),
        "resolution_source_field": market.get("resolutionSource", ""),
        "rules": rules,
        "rules_sha256": sha256_text(rules),
        "rule_template": template,
        "rule_template_sha256": sha256_text(template),
    }
    if not re.fullmatch(r"0x[0-9a-fA-F]{64}", row["condition_id"]):
        raise ValueError("Missing or malformed condition ID.")
    records.append(row)
    for outcome in ["yes", "no"]:
        token_records.append({
            "family": spec["family"], "deadline_date": spec["deadline_date"],
            "market_id": row["market_id"], "condition_id": row["condition_id"],
            "outcome": outcome.title(), "token_id": token_map[outcome],
        })

contracts = pd.DataFrame(records).sort_values(["family", "deadline_date"]).reset_index(drop=True)
tokens = pd.DataFrame(token_records)
for column in ["fetched_at_utc", "market_created_at_utc", "api_start_date_utc", "api_end_date_utc", "rule_cutoff_utc"]:
    contracts[column] = pd.to_datetime(contracts[column], utc=True, errors="raise", format="ISO8601")
if contracts[["market_created_at_utc", "api_start_date_utc", "api_end_date_utc"]].isna().any().any():
    raise ValueError("Missing metadata dates; do not guess the comparison window.")
if contracts["condition_id"].duplicated().any() or tokens["token_id"].duplicated().any():
    raise ValueError("Duplicate contracts or outcome token IDs.")
display(contracts[[
    "family", "deadline_date", "market_id", "market_created_at_utc",
    "api_start_date_utc", "rule_cutoff_utc", "api_end_date_utc", "closed_at_snapshot",
]])
display(tokens)


,family,deadline_date,market_id,market_created_at_utc,api_start_date_utc,rule_cutoff_utc,api_end_date_utc,closed_at_snapshot
0,hormuz_normalisation,2026-10-31,3501950,2026-08-10 22:38:21.735368+00:00,2026-08-11 09:19:18.848969+00:00,NaT,2026-11-01 03:59:00+00:00,False
1,hormuz_normalisation,2026-11-30,3501951,2026-08-10 22:39:10.449073+00:00,2026-08-11 09:20:18.921762+00:00,NaT,2026-12-01 04:59:00+00:00,False
2,hormuz_normalisation,2026-12-31,2176270,2026-05-06 17:46:53.811882+00:00,2026-05-11 13:03:57.905316+00:00,NaT,2027-01-01 04:59:00+00:00,False
3,russia_ukraine_agreement,2026-10-31,2243896,2026-05-12 19:16:23.087816+00:00,2026-05-13 03:30:53.564000+00:00,2026-11-01 03:59:00+00:00,2026-11-01 03:59:00+00:00,False
4,russia_ukraine_agreement,2026-11-30,4338145,2026-09-07 15:51:10.861043+00:00,2026-09-07 15:51:23+00:00,2026-12-01 04:59:00+00:00,2026-12-01 04:59:00+00:00,False
5,russia_ukraine_agreement,2026-12-31,2243897,2026-05-12 19:16:23.390053+00:00,2026-05-13 03:30:47.258382+00:00,2027-01-01 04:59:00+00:00,2027-01-01 04:59:00+00:00,False


,family,deadline_date,market_id,condition_id,outcome,token_id
0,hormuz_normalisation,2026-10-31,3501950,0x405b56fc75f56d7635a94df9836d6cb6f9b9571d1fd33eb165c1e59d1d11801f,Yes,65318042793305154928481822061228422214615698483328483747495637152414701080541
1,hormuz_normalisation,2026-10-31,3501950,0x405b56fc75f56d7635a94df9836d6cb6f9b9571d1fd33eb165c1e59d1d11801f,No,112428165813371561912833460698436648539922320261306010177252036814838759682453
2,hormuz_normalisation,2026-11-30,3501951,0xd64b917e1f7cdf1ad3f51326cf8985ce80e140291d2eaf2bf62821ef02231550,Yes,18077313030201522388676673488157544588986077823815688276055531102255531898625
3,hormuz_normalisation,2026-11-30,3501951,0xd64b917e1f7cdf1ad3f51326cf8985ce80e140291d2eaf2bf62821ef02231550,No,106375876055695859045413141458606643340242853794519126089005356920987236120368
4,hormuz_normalisation,2026-12-31,2176270,0x5c79dfde05559b79a9cb9f7c4187e4d49632dd042572ae676952f812732591cc,Yes,83783489454659420807679723574840032660002326710045123865554753994450797728825
5,hormuz_normalisation,2026-12-31,2176270,0x5c79dfde05559b79a9cb9f7c4187e4d49632dd042572ae676952f812732591cc,No,48727431314333088243693016929623363710488980152833544156961659445129455301311
6,russia_ukraine_agreement,2026-10-31,2243896,0x6c3f1009a0c91ba9f7aae44aeadf86863b0d433b60f7a090fd00c945019aa32f,Yes,10861802947610899379330568332287100318869035186187733531205554992019689450318
7,russia_ukraine_agreement,2026-10-31,2243896,0x6c3f1009a0c91ba9f7aae44aeadf86863b0d433b60f7a090fd00c945019aa32f,No,39319769648132414476170103739900770633009012502056309868976886150530270219889
8,russia_ukraine_agreement,2026-11-30,4338145,0x346543618b5077cedcabb9da57cf643922dbc752a7f332fc26e0256a38bb74c7,Yes,92891603310020284971853769274013951144660180302451630607247552989141667117598
9,russia_ukraine_agreement,2026-11-30,4338145,0x346543618b5077cedcabb9da57cf643922dbc752a7f332fc26e0256a38bb74c7,No,13274969701074786593458491024179777953579994335319473707625563805797711139412


In [5]:
# Expand any contract to inspect its complete, unabridged resolution text.
for row in contracts.to_dict("records"):
    display(HTML(
        "<details><summary><b>" + html.escape(row["question"]) + "</b></summary>"
        "<p><a href='" + html.escape(row["market_url"], quote=True) + "'>Market page</a></p>"
        "<pre style='white-space:pre-wrap'>" + html.escape(row["rules"]) + "</pre>"
        "</details>"
    ))


## 4. Human rules review — what these markets actually measure

**Hormuz:** IMF Portwatch's seven-day moving average of transit calls must reach **60 or more** on a qualifying date. This is not a government declaration that the strait has reopened. Each observation window starts at **that market's creation**. December was created earlier than October/November. Publication delays (up to 14 days after the final date), revisions and error-correction clauses matter. Similar wording alone does not prove identical settlement behavior.

**Russia–Ukraine:** a qualifying **mutually agreed, general suspension of direct military engagement** must be agreed by the deadline. The agreement can take effect later. An informal/unilateral pause or a pause limited to certain regions or target categories does not qualify. We must not mix these contracts with different markets asking when fighting actually stops.

The review below is tied to a fingerprint of the inspected text. If a refreshed response changes any rule except the explicit deadline date/whitespace, the notebook requires another review. Equal fingerprints are a useful check, **not a proof of arbitrage**.


In [6]:
# Pinned after inspecting the full source descriptions, not learned on each run.
REVIEWED_TEMPLATE_HASHES = {
    "hormuz_normalisation": "62d39877bac4bdacd8a067e48d779604d7f2d63bdfe80d750113000762e25e16",
    "russia_ukraine_agreement": "30d11234d6659d44f99791b64248a5ce464be9e75dde66455bf8851772a318be",
}
REVIEW_NOTES = {
    "hormuz_normalisation": {
        "criterion": "IMF Portwatch 7-day average transit calls >= 60 on a qualifying date.",
        "source": "IMF Portwatch, including downloadable transit data.",
        "time_basis": "Observation date; exact intraday cutoff not explicit. Grace/error clauses use ET.",
        "caution": "Different creation-based observation windows; publication and revision clauses. Not certified as a strict nested-event arbitrage.",
        "disposition": "Exploratory candidate with observation-window caveats.",
    },
    "russia_ukraine_agreement": {
        "criterion": "Qualifying general ceasefire AGREEMENT reached, not necessarily already effective.",
        "source": "Official Russian/Ukrainian government information and consensus credible reporting.",
        "time_basis": "11:59 PM America/New_York on the named deadline date.",
        "caution": "November listed later. Use only overlapping observed history; no claim of executable trading edge.",
        "disposition": "Comparable deadline definitions for an exploratory curve/lead-lag study.",
    },
}
contracts["rule_template_reviewed"] = [
    row.rule_template_sha256 == REVIEWED_TEMPLATE_HASHES.get(row.family)
    for row in contracts.itertuples()
]

family_rows = []
for family, group in contracts.groupby("family", sort=True):
    family_rows.append({
        "family": family,
        "contracts": len(group),
        "matching_rule_templates": group["rule_template_sha256"].nunique() == 1,
        "reviewed_text_unchanged": bool(group["rule_template_reviewed"].all()),
        # This is a metadata-based earliest possible common start, NOT evidence
        # that historical prices actually exist continuously from this time.
        "metadata_common_start_utc": group["api_start_date_utc"].max(),
        "earliest_deadline_date": group["deadline_date"].min(),
        "all_open_at_snapshot": bool(group["closed_at_snapshot"].eq(False).all()),
        **REVIEW_NOTES[family],
    })
families = pd.DataFrame(family_rows)
display(families[[
    "family", "contracts", "matching_rule_templates", "reviewed_text_unchanged",
    "metadata_common_start_utc", "disposition",
]])
for row in families.to_dict("records"):
    print("\n" + row["family"])
    print("Measures:", row["criterion"])
    print("Time basis:", row["time_basis"])
    print("Caution:", row["caution"])
if not contracts["rule_template_reviewed"].all():
    print("\nSTOP: rule text changed or is unreviewed. Read the full rules before updating a pinned hash.")


,family,contracts,matching_rule_templates,reviewed_text_unchanged,metadata_common_start_utc,disposition
0,hormuz_normalisation,3,True,True,2026-08-11 09:20:18.921762+00:00,Exploratory candidate with observation-window caveats.
1,russia_ukraine_agreement,3,True,True,2026-09-07 15:51:23+00:00,Comparable deadline definitions for an exploratory curve/lead-lag study.



hormuz_normalisation
Measures: IMF Portwatch 7-day average transit calls >= 60 on a qualifying date.
Time basis: Observation date; exact intraday cutoff not explicit. Grace/error clauses use ET.
Caution: Different creation-based observation windows; publication and revision clauses. Not certified as a strict nested-event arbitrage.

russia_ukraine_agreement
Measures: Qualifying general ceasefire AGREEMENT reached, not necessarily already effective.
Time basis: 11:59 PM America/New_York on the named deadline date.
Caution: November listed later. Use only overlapping observed history; no claim of executable trading edge.


## 5. Pair-by-pair audit

For a true same-event **happens by** family, later-deadline probabilities should be no lower. But we must examine the underlying observation windows and settlement clauses first.

Below, `creation_window_containment` is only a check on creation timestamps for Hormuz; even a True value does not certify settlement equivalence. Ukraine does not explicitly define its qualifying window as starting at market creation, so that field is not applicable there.

The earliest shared trading start comes from metadata. In step 2, actual observed price coverage must determine the usable window. We will neither backfill a contract before it existed nor treat missing observations as zero volume.


In [7]:
pair_rows = []
for family, group in contracts.groupby("family", sort=True):
    ordered = group.sort_values("deadline_date").to_dict("records")
    for early, late in combinations(ordered, 2):
        creation_containment = None
        if family == "hormuz_normalisation":
            creation_containment = late["market_created_at_utc"] <= early["market_created_at_utc"]
        pair_rows.append({
            "family": family,
            "earlier_market_id": early["market_id"],
            "later_market_id": late["market_id"],
            "earlier_deadline": early["deadline_date"],
            "later_deadline": late["deadline_date"],
            "same_rule_template": early["rule_template_sha256"] == late["rule_template_sha256"],
            "reviewed_text_unchanged": early["rule_template_reviewed"] and late["rule_template_reviewed"],
            "different_creation_times": early["market_created_at_utc"] != late["market_created_at_utc"],
            "creation_window_containment": creation_containment,
            "metadata_common_start_utc": max(early["api_start_date_utc"], late["api_start_date_utc"]),
            "nesting_certified": False,
            "review_note": REVIEW_NOTES[family]["caution"],
        })
pairs = pd.DataFrame(pair_rows)
display(pairs[[
    "family", "earlier_deadline", "later_deadline", "same_rule_template",
    "creation_window_containment", "metadata_common_start_utc", "nesting_certified",
]])
print("No pair is certified here as risk-free arbitrage or as a profitable lead-lag trade.")


,family,earlier_deadline,later_deadline,same_rule_template,creation_window_containment,metadata_common_start_utc,nesting_certified
0,hormuz_normalisation,2026-10-31,2026-11-30,True,False,2026-08-11 09:20:18.921762+00:00,False
1,hormuz_normalisation,2026-10-31,2026-12-31,True,True,2026-08-11 09:19:18.848969+00:00,False
2,hormuz_normalisation,2026-11-30,2026-12-31,True,True,2026-08-11 09:20:18.921762+00:00,False
3,russia_ukraine_agreement,2026-10-31,2026-11-30,True,None,2026-09-07 15:51:23+00:00,False
4,russia_ukraine_agreement,2026-10-31,2026-12-31,True,None,2026-05-13 03:30:53.564000+00:00,False
5,russia_ukraine_agreement,2026-11-30,2026-12-31,True,None,2026-09-07 15:51:23+00:00,False


No pair is certified here as risk-free arbitrage or as a profitable lead-lag trade.


## 6. Validate before saving

These small checks run offline and exercise the outcome mapping, date handling and rule-change detection. They do not replace the manual rule review.


In [8]:
assert len(contracts) == 6 and len(tokens) == 12 and len(pairs) == 6
assert contracts["market_id"].is_unique and tokens["token_id"].is_unique
assert contracts.groupby("family").size().eq(3).all()
assert contracts["rules"].str.len().gt(100).all()
assert outcome_tokens('["No", "Yes"]', '["111", "222"]') == {"no": "111", "yes": "222"}
assert outcome_tokens(["Yes", "No"], ["222", "111"])["yes"] == "222"
for bad_outcomes, bad_tokens in [
    (["Yes", "Yes"], ["111", "222"]),
    (["Yes", "No"], [111, 222]),
    (["Yes", "No"], ["111"]),
    (["Yes", "No"], ["111", "111"]),
]:
    try:
        outcome_tokens(bad_outcomes, bad_tokens)
    except ValueError:
        pass
    else:
        raise AssertionError("Malformed outcome mapping was accepted")
assert rule_template("Threshold 60 by October 31, 2026", "October 31, 2026") == "Threshold 60 by <DEADLINE>"
assert sha256_text(rule_template("Threshold 60 by October 31, 2026", "October 31, 2026")) != sha256_text(
    rule_template("Threshold 61 by October 31, 2026", "October 31, 2026")
)
ukraine_cutoffs = contracts.loc[
    contracts["family"].eq("russia_ukraine_agreement"), "rule_cutoff_utc"
]
assert [value.hour for value in ukraine_cutoffs] == [3, 4, 4]  # US daylight saving ends in November.
assert contracts.loc[contracts["family"].eq("hormuz_normalisation"), "rule_cutoff_utc"].isna().all()
if not contracts["rule_template_reviewed"].all():
    raise RuntimeError("Rules need review: exports blocked; do not blindly replace the fingerprints.")
print("Passed: exact six-contract selection, 12 unique tokens, date/DST handling and rule review.")


Passed: exact six-contract selection, 12 unique tokens, date/DST handling and rule review.


## 7. Save the research shortlist

Each successful run creates a new folder, so an old audit is not overwritten. Raw API snapshots and the unabridged rules are retained alongside the compact tables.

- `contracts.parquet` / `contracts.csv`: IDs, dates, complete rules and review fingerprints.
- `tokens.csv`: outcome-to-token mapping (use string dtype when reading CSV IDs).
- `families.csv` / `pairs.csv`: comparison windows and caveats.
- `rules.md`: human-readable full rules and review notes.
- `manifest.json`: provenance, artifact hashes, and explicitly completed/not-started stages.

Prefer Parquet/JSON for IDs: spreadsheet software can round long numeric-looking CSV fields. `latest.json` points to this run. Files under `data/` are already Git-ignored.


In [9]:
if not contracts["rule_template_reviewed"].all():
    raise RuntimeError("Cannot export an unreviewed shortlist.")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
RUN = BASE / "runs" / run_id
RUN.mkdir(parents=True, exist_ok=False)
raw_dir = RUN / "raw"
raw_dir.mkdir()
for slug, snapshot in snapshots.items():
    atomic_json(raw_dir / f"{slug}.json", snapshot)

export_contracts = contracts.drop(columns=["rule_template"])
export_contracts.to_parquet(RUN / "contracts.parquet", index=False)
export_contracts.to_csv(RUN / "contracts.csv", index=False, encoding="utf-8-sig")
tokens.to_csv(RUN / "tokens.csv", index=False, encoding="utf-8-sig")
families.to_csv(RUN / "families.csv", index=False, encoding="utf-8-sig")
pairs.to_csv(RUN / "pairs.csv", index=False, encoding="utf-8-sig")

rules_sections = ["# Deadline-market contract audit\n", "Metadata/rules only. No trading or price-history analysis.\n"]
for row in contracts.to_dict("records"):
    rules_sections.extend([
        f"## {row['question']}\n",
        f"Source: {row['market_url']}\n",
        f"Retrieved: {row['fetched_at_utc'].isoformat()}\n",
        f"Condition ID: {row['condition_id']}\n",
        f"Review caveat: {REVIEW_NOTES[row['family']]['caution']}\n",
        row["rules"] + "\n",
    ])
(RUN / "rules.md").write_text("\n".join(rules_sections), encoding="utf-8")

artifacts = {}
for path in sorted(RUN.rglob("*")):
    if path.is_file():
        artifacts[path.relative_to(RUN).as_posix()] = {
            "bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
manifest = {
    "schema_version": 1,
    "run_id": run_id,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "contract_count": len(contracts),
    "outcome_token_count": len(tokens),
    "event_count": len(snapshots),
    "stages": {
        "metadata_and_rules": "complete",
        "manual_rule_review": "complete_with_documented_caveats",
        "price_history": "not_started",
        "interval_volume": "not_started",
        "curve_and_lead_lag_tests": "not_started",
        "executable_profitability": "not_tested",
    },
    "nesting_or_arbitrage_certified": False,
    "reviewed_template_hashes": REVIEWED_TEMPLATE_HASHES,
    "sources": [
        {"event_slug": slug, "source_url": item["source_url"], "fetched_at_utc": item["fetched_at_utc"]}
        for slug, item in snapshots.items()
    ],
    "artifacts": artifacts,
}
atomic_json(RUN / "manifest.json", manifest)
atomic_json(BASE / "latest.json", {"run_id": run_id, "run_path": RUN.relative_to(ROOT).as_posix()})
print("Saved:", RUN)
print("Step 1 complete: 6 contracts / 12 outcome tokens / 4 source event records.")
print("Price histories and interval volumes have NOT been downloaded by this notebook.")


Saved: c:\Users\kpast\OneDrive\Dokumenty\Polytrader\polytrader\data\research\deadline_markets\step1\runs\20261003T134136Z-e0c5562f
Step 1 complete: 6 contracts / 12 outcome tokens / 4 source event records.
Price histories and interval volumes have NOT been downloaded by this notebook.


## What we know now, and what comes next

We have a reproducible six-contract shortlist with saved rules and the token IDs needed for data collection.

- **Ukraine:** the three descriptions use the same agreement definition. November's later listing restricts the common history window.
- **Hormuz:** the threshold/source match, but creation-based windows and publication/revision clauses need to remain visible in every interpretation. A price ordering violation is not automatically an arbitrage opportunity.
- Blank Gamma `resolutionSource` fields do not mean there is no source: the authoritative source is described in the full rules.
- Saved rules describe the current snapshot. They do **not** establish which wording traders saw at every historical timestamp.

**Step 2 (not run here):** pull overlapping prices for these exact token IDs, inspect actual timestamps/resolution/gaps, then obtain interval volume if available. Only after those checks should we fit deadline curves or test whether one contract leads another.
